In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

In [11]:
from sklearn.datasets import load_iris, fetch_california_housing

# Iris dataset (classification)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# California Housing dataset (regression)
housing = fetch_california_housing()
X_house, y_house = housing.data, housing.target

In [12]:
# Classification split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# Regression split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_house, y_house, test_size=0.3, random_state=42)

In [13]:
clf = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)
clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)

print("Classification Accuracy:", accuracy_score(y_test_c, y_pred_c))
print(classification_report(y_test_c, y_pred_c))
print(export_text(clf, feature_names=iris.feature_names))

Classification Accuracy: 0.9777777777777777
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.92      0.96        13
           2       0.93      1.00      0.96        13

    accuracy                           0.98        45
   macro avg       0.98      0.97      0.97        45
weighted avg       0.98      0.98      0.98        45

|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal length (cm) <= 4.75
|   |   |--- petal width (cm) <= 1.60
|   |   |   |--- class: 1
|   |   |--- petal width (cm) >  1.60
|   |   |   |--- class: 2
|   |--- petal length (cm) >  4.75
|   |   |--- petal length (cm) <= 5.15
|   |   |   |--- class: 2
|   |   |--- petal length (cm) >  5.15
|   |   |   |--- class: 2



In [14]:
reg = DecisionTreeRegressor(criterion="squared_error", max_depth=5, random_state=42)
reg.fit(X_train_r, y_train_r)

y_pred_r = reg.predict(X_test_r)

print("Regression MSE:", mean_squared_error(y_test_r, y_pred_r))
print("Regression R2 Score:", r2_score(y_test_r, y_pred_r))

Regression MSE: 0.5210801561811793
Regression R2 Score: 0.6029986793705844


In [16]:
# Step 6: Test with Multiple Datasets (Titanic + Ames Housing)

import seaborn as sns
from sklearn.datasets import fetch_openml

# Titanic dataset (classification)
titanic = sns.load_dataset("titanic")
titanic = titanic[["sex","age","fare","class","embarked","survived"]].dropna()
titanic = pd.get_dummies(titanic, columns=["sex","class","embarked"], drop_first=True)

X_titanic = titanic.drop("survived", axis=1)
y_titanic = titanic["survived"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_titanic, y_titanic, test_size=0.3, random_state=42)

clf_titanic = DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=42)
clf_titanic.fit(X_train_t, y_train_t)
y_pred_t = clf_titanic.predict(X_test_t)

print("Titanic Classification Accuracy:", accuracy_score(y_test_t, y_pred_t))
print(classification_report(y_test_t, y_pred_t))

# Ames Housing dataset (regression)
ames = fetch_openml(name="house_prices", as_frame=True)
X_ames = ames.data.select_dtypes(include=[np.number]).dropna(axis=1)  # keep numeric features only
y_ames = ames.target.astype(float)

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_ames, y_ames, test_size=0.3, random_state=42)

reg_ames = DecisionTreeRegressor(criterion="squared_error", max_depth=5, random_state=42)
reg_ames.fit(X_train_a, y_train_a)
y_pred_a = reg_ames.predict(X_test_a)

print("Ames Housing Regression MSE:", mean_squared_error(y_test_a, y_pred_a))
print("Ames Housing Regression R2 Score:", r2_score(y_test_a, y_pred_a))

Titanic Classification Accuracy: 0.780373831775701
              precision    recall  f1-score   support

           0       0.74      0.95      0.83       122
           1       0.89      0.55      0.68        92

    accuracy                           0.78       214
   macro avg       0.82      0.75      0.76       214
weighted avg       0.81      0.78      0.77       214

Ames Housing Regression MSE: 1654542301.6097164
Ames Housing Regression R2 Score: 0.7628948021777509


In [17]:
results = pd.DataFrame({
    "Dataset": ["Iris", "Titanic", "California Housing", "Ames Housing"],
    "Model Type": ["Classification", "Classification", "Regression", "Regression"],
    "Metric": ["Accuracy", "Accuracy", "R² Score", "R² Score"],
    "Value": [
        accuracy_score(y_test_c, y_pred_c),
        accuracy_score(y_test_t, y_pred_t),
        r2_score(y_test_r, y_pred_r),
        r2_score(y_test_a, y_pred_a)
    ]
})

print(results)

              Dataset      Model Type    Metric     Value
0                Iris  Classification  Accuracy  0.977778
1             Titanic  Classification  Accuracy  0.780374
2  California Housing      Regression  R² Score  0.602999
3        Ames Housing      Regression  R² Score  0.762895


In [18]:
from sklearn.tree import export_text

# Train the classifier (already done earlier)
clf_titanic = DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=42)
clf_titanic.fit(X_train_t, y_train_t)

# Export the tree as text
tree_rules_titanic = export_text(clf_titanic, feature_names=list(X_titanic.columns))
print(tree_rules_titanic)

|--- sex_male <= 0.50
|   |--- class_Third <= 0.50
|   |   |--- fare <= 29.36
|   |   |   |--- fare <= 28.23
|   |   |   |   |--- class: 1
|   |   |   |--- fare >  28.23
|   |   |   |   |--- class: 0
|   |   |--- fare >  29.36
|   |   |   |--- class: 1
|   |--- class_Third >  0.50
|   |   |--- fare <= 23.09
|   |   |   |--- age <= 1.50
|   |   |   |   |--- class: 1
|   |   |   |--- age >  1.50
|   |   |   |   |--- class: 0
|   |   |--- fare >  23.09
|   |   |   |--- fare <= 31.33
|   |   |   |   |--- class: 0
|   |   |   |--- fare >  31.33
|   |   |   |   |--- class: 0
|--- sex_male >  0.50
|   |--- fare <= 15.72
|   |   |--- age <= 13.50
|   |   |   |--- class: 1
|   |   |--- age >  13.50
|   |   |   |--- fare <= 11.00
|   |   |   |   |--- class: 0
|   |   |   |--- fare >  11.00
|   |   |   |   |--- class: 0
|   |--- fare >  15.72
|   |   |--- class_Third <= 0.50
|   |   |   |--- age <= 13.00
|   |   |   |   |--- class: 1
|   |   |   |--- age >  13.00
|   |   |   |   |--- class: 0
|  